# NEU-DET Dataset Exploration

This notebook explores the NEU-DET steel surface defect dataset after conversion to YOLO format.

**Dataset**: NEU-DET (1770 images, 6 defect classes)
**Source**: https://github.com/siddhartamukherjee/NEU-DET-Steel-Surface-Defect-Detection
**Classes**: crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import yaml
import json
import random
from collections import Counter
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

# Load dataset config
with open("configs/dataset.yaml", "r") as f:
    config = yaml.safe_load(f)

neu_config = config["neu_det"]
class_names = neu_config["classes"]
print(f"Classes: {class_names}")
print(f"Number of classes: {len(class_names)}")

In [ ]:
# Load YOLO dataset config
with open("data/processed/neu_det/dataset.yaml", "r") as f:
    yolo_config = yaml.safe_load(f)

print(f"YOLO config:")
print(f"  Path: {yolo_config['path']}")
print(f"  Train: {yolo_config['train']}")
print(f"  Val: {yolo_config['val']}")
print(f"  Test: {yolo_config['test']}")
print(f"  nc: {yolo_config['nc']}")
print(f"  names: {yolo_config['names']}")

In [ ]:
# Count images and labels per split
base = Path("data/processed/neu_det")
splits = ["train", "val", "test"]

for split in splits:
    img_dir = base / "images" / split
    lbl_dir = base / "labels" / split
    n_img = len(list(img_dir.glob("*.jpg")))
    n_lbl = len(list(lbl_dir.glob("*.txt")))
    print(f"{split:5s}: {n_img:4d} images, {n_lbl:4d} label files")

In [ ]:
# Analyze class distribution across splits
class_counts = {split: Counter() for split in splits}
total_objects = {split: 0 for split in splits}

for split in splits:
    lbl_dir = base / "labels" / split
    for lbl_file in lbl_dir.glob("*.txt"):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    class_id = int(parts[0])
                    class_counts[split][class_id] += 1
                    total_objects[split] += 1

print("Class distribution per split:")
print(f"{'Split':<8} {'Total':>6}", end="")
for cid in range(len(class_names)):
    print(f" {class_names[cid]:>15}", end="")
print()

for split in splits:
    print(f"{split:<8} {total_objects[split]:>6}", end="")
    for cid in range(len(class_names)):
        print(f" {class_counts[split][cid]:>15}", end="")
    print()

In [ ]:
# Analyze bounding box statistics
bbox_stats = {cid: {"w": [], "h": [], "area": [], "aspect": []} for cid in range(len(class_names))}

for split in splits:
    lbl_dir = base / "labels" / split
    for lbl_file in lbl_dir.glob("*.txt"):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    class_id = int(parts[0])
                    w = float(parts[3])
                    h = float(parts[4])
                    bbox_stats[class_id]["w"].append(w)
                    bbox_stats[class_id]["h"].append(h)
                    bbox_stats[class_id]["area"].append(w * h)
                    bbox_stats[class_id]["aspect"].append(w / h if h > 0 else 0)

print("Bounding box statistics (normalized 0-1):")
print(f"{'Class':<20} {'Count':>6} {'Mean W':>8} {'Mean H':>8} {'Mean Area':>10} {'Mean Aspect':>10}")
for cid in range(len(class_names)):
    n = len(bbox_stats[cid]["w"])
    if n > 0:
        print(f"{class_names[cid]:<20} {n:>6} {np.mean(bbox_stats[cid]['w']):>8.4f} {np.mean(bbox_stats[cid]['h']):>8.4f} {np.mean(bbox_stats[cid]['area']):>10.6f} {np.mean(bbox_stats[cid]['aspect']):>10.2f}")

In [ ]:
# Visualize sample images with bounding boxes
def draw_bboxes(image_path, label_path, class_names):
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                class_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:])
                
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                
                color = plt.cm.tab10(class_id % 10)
                color = tuple(int(c * 255) for c in color[:3])
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, class_names[class_id], (x1, max(y1 - 5, 15)),
                          cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

# Show 3 samples per class
fig, axes = plt.subplots(6, 3, figsize=(12, 20))

for cid, class_name in enumerate(class_names):
    # Find images with this class
    class_images = []
    for split in splits:
        lbl_dir = base / "labels" / split
        for lbl_file in lbl_dir.glob("*.txt"):
            with open(lbl_file) as f:
                first_line = f.readline().strip()
                if first_line and int(first_line.split()[0]) == cid:
                    img_file = base / "images" / split / (lbl_file.stem + ".jpg")
                    if img_file.exists():
                        class_images.append((img_file, lbl_file))
                    break
        if len(class_images) >= 3:
            break
    
    for i in range(min(3, len(class_images))):
        img_file, lbl_file = class_images[i]
        img = draw_bboxes(img_file, lbl_file, class_names)
        axes[cid, i].imshow(img)
        axes[cid, i].set_title(f"{class_name} - {img_file.name}")
        axes[cid, i].axis('off')
    
    if len(class_images) == 0:
        for i in range(3):
            axes[cid, i].text(0.5, 0.5, "No samples found", ha='center', va='center')
            axes[cid, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Image properties analysis
sample_img = base / "images" / "train" / "crazing_1.jpg"
img = cv2.imread(str(sample_img))
print(f"Image shape: {img.shape}")
print(f"Image dtype: {img.dtype}")
print(f"Min/Max pixel values: {img.min()}/{img.max()}")
print(f"Mean pixel value: {img.mean():.2f}")
print(f"Std pixel value: {img.std():.2f}")

# Check a few more images for consistency
shapes = []
for split in splits:
    img_dir = base / "images" / split
    for img_file in list(img_dir.glob("*.jpg"))[:10]:
        img = cv2.imread(str(img_file))
        shapes.append(img.shape)

unique_shapes = set(shapes)
print(f"\nUnique image shapes in dataset: {unique_shapes}")

In [ ]:
# Class balance visualization
total_per_class = Counter()
for split in splits:
    total_per_class.update(class_counts[split])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
classes = [class_names[i] for i in range(len(class_names))]
counts = [total_per_class[i] for i in range(len(class_names))]
colors = plt.cm.tab10(range(len(class_names)))

ax1.bar(classes, counts, color=colors)
ax1.set_title("Total Object Count per Class (All Splits)")
ax1.set_ylabel("Number of Objects")
ax1.tick_params(axis='x', rotation=45)

# Pie chart
ax2.pie(counts, labels=classes, autopct='%1.1f%%', colors=colors)
ax2.set_title("Class Distribution (Percentage)")

plt.tight_layout()
plt.show()

print("Class balance summary:")
for cid, name in enumerate(class_names):
    pct = 100 * total_per_class[cid] / sum(counts)
    print(f"  {name:<20}: {total_per_class[cid]:>4} ({pct:.1f}%)")

In [ ]:
# Bounding box size distribution per class
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for cid, class_name in enumerate(class_names):
    areas = bbox_stats[cid]["area"]
    if areas:
        axes[cid].hist(areas, bins=30, alpha=0.7, color=plt.cm.tab10(cid), edgecolor='black')
        axes[cid].set_title(f"{class_name} - BBox Area Distribution")
        axes[cid].set_xlabel("Normalized Area")
        axes[cid].set_ylabel("Frequency")
        axes[cid].axvline(np.mean(areas), color='red', linestyle='--', label=f'Mean: {np.mean(areas):.4f}')
        axes[cid].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Aspect ratio distribution per class
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for cid, class_name in enumerate(class_names):
    aspects = bbox_stats[cid]["aspect"]
    if aspects:
        axes[cid].hist(aspects, bins=30, alpha=0.7, color=plt.cm.tab10(cid), edgecolor='black')
        axes[cid].set_title(f"{class_name} - Aspect Ratio (W/H) Distribution")
        axes[cid].set_xlabel("Aspect Ratio")
        axes[cid].set_ylabel("Frequency")
        axes[cid].axvline(np.mean(aspects), color='red', linestyle='--', label=f'Mean: {np.mean(aspects):.2f}')
        axes[cid].axvline(1.0, color='green', linestyle=':', label='Square (1.0)')
        axes[cid].legend()
        axes[cid].set_xlim(0, max(5, np.max(aspects)))

plt.tight_layout()
plt.show()

## Summary

The NEU-DET dataset has been successfully prepared for YOLO training:

- **Total images**: 1770
- **Train/Val/Test split**: 1239 / 265 / 266 (70%/15%/15%)
- **Classes**: 6 (crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches)
- **Total annotations**: 4126 bounding boxes
- **Image resolution**: 200x200 pixels (native)
- **Annotation format**: YOLO (normalized center-x, center-y, width, height)

### Key Observations:
1. Class distribution is somewhat imbalanced (inclusion has the most, pitted_surface the fewest)
2. All images are consistent 200x200 resolution
3. Bounding boxes are generally large relative to image size (defects cover significant portions)
4. Aspect ratios vary by defect type - some are more elongated (scratches) others more square

### Next Steps:
1. Train YOLOv8 detection model using `configs/detection.yaml`
2. Train YOLOv8-seg segmentation model using `configs/segmentation.yaml`
3. Evaluate on test split and compare detection vs segmentation performance